In [ ]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.vectorstores.pgvector import PGVector
from langchain_community.embeddings import GPT4AllEmbeddings
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

from langchain_openai import ChatOpenAI

from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import dotenv_values

env_values = dotenv_values(".env")
API_KEY = env_values["API_KEY"]
API_ENDPOINT = env_values["API_ENDPOINT"]
PG_PASS = env_values["POSTGRES_PASSWORD"]

CONNECTION_STRING = "postgresql+psycopg2://postgres:" + PG_PASS + "@pgvector:5432/postgres"
COLLECTION_NAME = 'embedding'

loader = DirectoryLoader(
    './docs/', 
    glob="**/*.txt", 
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)

documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
texts = text_splitter.split_documents(documents)

embeddings = GPT4AllEmbeddings()

db = PGVector.from_documents(
    embedding=embeddings,
    documents=texts,
    collection_name=COLLECTION_NAME,
    connection_string=CONNECTION_STRING,
)

query = "Tell me what you think the most interesting cat fact is."
retriever = db.as_retriever(search_kwargs={'k': 50})
retriever.invoke(query)

claude_model = ChatOpenAI(
    model="ai2s-claude-sonnet-4-6",
    openai_api_base=API_ENDPOINT,
    openai_api_key=API_KEY,
    temperature=0.2
)

qa = RetrievalQA.from_chain_type(llm=claude_model, chain_type="stuff", retriever=retriever)
qa.run(query)
